
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Agent Bricks와 함께하는 지식 어시스턴트

## 서론

이 강의에서는 Databricks Mosaic AI 내의 선언적 Framework인 **Agent Bricks**를 소개하며, 프로덕션 준비 완료 AI 에이전트 생성을 단순화합니다. 우리는 특히 **Knowledge Assistant**에 집중할 예정이며, 이는 기업 문서에 기반한 전문 대화 에이전트를 구축하는 패턴입니다. Agent Bricks가 개발 패러다임을 수동 조정에서 결과 지향적 선언으로 전환하는 과정을 배우게 되며, 자동화된 최적화 루프를 활용해 효율성을 높이는 방법을 알 수 있습니다. 마지막으로, 이 에이전트들을 구동하는 기본 아키텍처를 탐구하여 견고하고 확장 가능하며 거버넌스가 잘 관리되도록 할 것입니다.

## 수업 목표

이 수업이 끝날 때쯤이면 다음과 같은 행동을 할 수 있게 될 것입니다:

* Agent Bricks의 핵심 가치 제안을 기존 수동 개발 방식과 비교하여 **정의**하십시오.  
* 정보 추출 및 맞춤형 대규모 언어 모델(LLM)을 포함한 Agent Bricks의 주요 사용 사례를 **파악**하십시오.  
* 구문 분석, AI Search, model serving 제공 등과 같은 지식 어시스턴트의 아키텍처 구성 요소를 **설명**하십시오. 
* “품질 루프”를 설명하고, 인간 피드백 기반 에이전트 학습(ALHF)이 에이전트 성능을 최적화하는 방식을 **기술**하십시오.

## A. Agent Bricks란 무엇인가요?

**Agent Bricks**는 Databricks Mosaic AI 내의 **선언형** Framework로, 생산 품질의 AI 에이전트의 생성, 배포 및 최적화를 가속화하도록 설계되었습니다. 전통적인 '직접 만들기'(DIY) 방식과 달리, 엔지니어가 수동으로 모델을 선택하고 청크 전략을 구성하며 프롬프트를 수동으로 조정해야 하는 반면, Agent Bricks는 제공된 데이터와 태스크를 바탕으로 이러한 구성 결정을 자동화합니다.

### A1. 생산 AI의 도전

생성형 AI를 개념 증명(PoC)에서 생산으로 전환하는 데는 세 가지 주요 마찰 지점이 있습니다:

1. **최적화 복잡도:** AI 시스템은 LLM 선택(예: Llama 4 vs. GPT-4o) 등 여러 가지 '노브'를 가지고 있습니다. 또한 청크 크기 및 임베딩 모델과 같은 검색 전략들, 프롬프트 엔지니어링 기법 등이 포함됩니다. 특정 기업 데이터세트에 최적의 조합을 찾는 데는 시간이 많이 소요됩니다.  
2. **평가 난이도:** 어떤 에이전트가 생산에 '충분히 좋은'지 판단하려면 엄격한 테스트가 필요합니다. 팀은 종종 라벨이 붙은 '골든 데이터세트'나 검증 가능한 지표가 없고, 대신 주관적인 '분위기 체크'에 의존합니다.  
3. **비용 vs. 품질 트레이드오프:** 높은 품질을 달성하려면 종종 고가의 대형 모델이 필요합니다. 비용을 줄이면 보통 성능이 저하됩니다. 팀은 가능한 한 낮은 비용으로 품질을 극대화하는 최적의 균형을 찾기 위해 고군분투합니다.

### A2. Agent Bricks 솔루션

Agent Bricks는 에이전트 정의를 **선언적**으로 처리하여 이러한 문제를 해결합니다. 데이터를 제공하고 태스크를 선택하면 Agent Bricks 엔진이 시스템을 반복적으로 최적화합니다.

이를 이끄는 핵심 메커니즘은 **인간 피드백을 통한 에이전트 학습(ALHF)** 입니다. 시스템:

1. 기준 에이전트를 즉시 **배포**합니다.  
2. 리뷰 앱을 통해 피드백을 **수집**합니다(좋아요/싫어요, 정정된 답변). 
3. 이 피드백을 종합하여 평가 기준을 자동 생성하고, 수동 코드 변경 없이 기본 프롬프트와 구성을 **최적화**합니다.

<!-- <img src="../Includes/images/05-agent-bricks-workflow.png" width="400"/> -->
![05-agent-bricks-workflow](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/05-agent-bricks-workflow.png)

*그림 1: Agent Bricks 최적화 사이클. 시스템은 작업 선언에서 배포로 전환한 후, 피드백을 활용해 최적화를 추진하며 지속적인 개선 루프를 형성합니다.*


## B. Agent Bricks 사용 사례

Agent Bricks는 일반적인 기업 패턴을 위한 사전 구성된 아키텍처("브릭")를 제공합니다. 각 브릭은 특정 상호작용 및 데이터 처리 방식에 특화되어 있습니다.

### B1. 지식 어시스턴트

이것이 이번 강의의 초점입니다. **지식 어시스턴트는 기업 문서를 전문가의 대화 에이전트로 바꿉니다**.

* **함수:** 그것은 지정된 파일에 대해 검색 증강 생성(RAG)을 수행합니다. 그것은 파싱, 청킹, 임베딩, 인용 생성을 자동으로 처리합니다.  
* **사용 사례:** 핸드북을 기반으로 정책 질문에 답변하는 HR 봇, 또는 제품 매뉴얼을 기반으로 티켓을 해결하는 기술 지원 봇.

### B2. 정보 추출

이 에이전트 유형은 비구조화된 문서(예: PDF, 이미지, 텍스트 파일)를 구조화된 데이터로 변환합니다.

* **함수:** 이 에이전트는 JSON 스키마에 의해 정의된 특정 필드를 추출합니다.  
* **사용 사례:** 송장 저장소를 "공급업체 이름", "총 금액", "날짜"가 포함된 구조화된 Delta 테이블로 변환하거나 법적 계약서에서 조항을 추출하는 것.

### B3. 다중 에이전트 감독자

이 고급 패턴은 여러 에이전트와 도구를 조율하여 복잡하고 다단계적인 문제를 해결합니다.

* **기능:** "감독자" 에이전트는 사용자 쿼리를 올바른 하위 에이전트나 도구(예: Unity Catalog 함수)로 라우팅합니다.  
* **사용 사례:** 감독자가 청구 질문을 **Genie** Space(구조화된 데이터)으로, 기술 문제 해결 질문을 **지식 어시스턴트**(비정형 데이터)로 라우팅하는 고객 지원 시스템입니다.

### B4. 커스텀 LLM

이 에이전트는 특정 기업 지침과 작업에 맞춘 특화된 LLM 엔드포인트를 생성합니다.

* **기능:** 이 모델을 특정 톤, 형식 또는 규정 준수 규칙에 맞게 최적화합니다.  
* **사용 사례:** 브랜드의 스타일 가이드를 엄격히 준수하는 소셜 미디어 게시물을 작성하는 마케팅 생성기, 또는 임원 보고서에 특정 형식을 출력하는 요약 도구.

## C. 선언적 대 코드 우선 방법

Databricks에서 AI 에이전트를 구축할 때, 개발자들은 일반적으로 두 가지 주요 추상화 수준 중 하나를 선택합니다: 코드 우선과 선언적.

### C1. 코드-퍼스트 (Mosaic AI 에이전트 프레임워크)

이 방법은 최대한의 제어력을 제공하지만 더 많은 노력이 필요합니다. 개발자들은 핵심 에이전트 로직을 코드(LangChain, LlamaIndex, OpenAI SDK) 같은 Python 라이브러리를 사용하며, **Mosaic AI 에이전트 프레임워크**를 사용해 스캐폴딩, 추적, 거버넌스를 수행합니다.

* **워크플로:** 개발자는 수동으로 검색 논리를 작성하고, 프롬프트 템플릿을 정의하며, 임베딩 모델을 선택하고, AI Search 인덱스 동기화를 관리합니다. 그들은 에이전트 Framework를 사용해 MLflow로 추적을 로그하고 에이전트를 Model Serving 엔드포인트로 배포합니다.  
* **장점:** 무한한 맞춤화 가능성. 새로운 추론 루프나 매우 구체적인 도구 사용을 구현할 수 있습니다.  
* **단점:** 개발자가 기술 부채를 책임집니다. 최적화(청킹, 프롬프트)는 수작업이며, 검색 전략이 변경되어야 할 경우(예: 청크 크기 변경) 코드를 다시 작성하고 재배포해야 합니다.

### C2. 선언문(Agent Bricks)

이것이 바로 '결과 지향적' 접근법입니다. 개발자는 에이전트가 무엇을 해야 하는지를 선언하는 것이지, 어떻게 해야 하는지는 선언하지 않습니다.

* **워크플로:** 개발자는 "Knowledge Assistant"를 선택하고 PDF가 포함된 Unity Catalog 볼륨을 가리키며, 페르소나에 대한 텍스트 설명을 제공합니다. 에이전트 브릭스는 파싱, 인덱싱, 프롬프트 엔지니어링을 담당합니다.  
* **장점:** 가장 빠른 가치 실현 시간. 시스템은 합성 데이터를 생성하여 스스로 테스트하고, 피드백을 기반으로 자동 최적화합니다.  
* **단점:** 저수준 실행 로직에 대한 세밀한 제어가 순수 코드보다 부족합니다.


## D. 지식 보조 구성 요소

에이전트 브릭스로 만든 지식 보조는 '블랙박스'가 아닙니다; 이는 네이티브 Databricks 아키텍처로 구성된 시스템입니다. 이러한 구성 요소를 이해하는 것은 디버깅과 거버넌스에 매우 중요합니다.

<!-- <img src="../Includes/images/05-agent-bricks-components.png"/> -->

![05-agent-bricks-components](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/05-agent-bricks-components.png)

*그림 2: 에이전트 브릭스 지식 어시스턴트 구성 요소. 이 구성 요소들은 백그라운드에서 작동하며 사용자가 직접 관리할 필요가 없습니다.*


### D1. 데이터 수집 및 파싱

지식 보조기의 기초는 **Unity Catalog 볼륨**에 저장된 데이터에 있습니다.

* **출처:** 사용자는 파일(PDF, DOCX, HTML)을 포함하는 볼륨을 선택합니다.  
* **구문 분석:** 시스템은 복잡한 문서에서 텍스트, 표, 이미지를 추출하기 위해 **ai\_parse\_document**, AI 함수를 사용합니다. 이렇게 하면 PDF 내 시각적 요소들(예: 차트)이 LLM이 이해할 수 있는 맥락으로 변환됩니다.

### D2. Databricks AI Search

파싱이 완료되면 데이터를 검색하기 위해 인덱싱되어야 합니다.

* **관리형 임베딩:** 에이전트 브릭스는 임베딩 모델(예: GTE)을 자동으로 선택하고 **Databricks AI Search** 인덱스를 제공합니다.  
* **동기화:** 인덱스가 완전히 관리됩니다. 새 파일이 소스 볼륨에 추가되면 AI Search 인덱스가 자동으로 업데이트되어 에이전트가 수동 재인덱싱 없이 항상 최신 정보를 유지할 수 있도록 합니다.

### D3. 추론 엔진과 Model Serving

에이전트 로직은 **Model Serving**에 호스팅됩니다.

* **추론:** 사용자가 질문을 할 때, 시스템은 쿼리를 벡터로 변환하고, AI Search에서 관련 청크를 가져와 LLM에 전달합니다.  
* **인용:** 중요한 점은, 지식 보조는 인용을 제공하도록 설계되어 있다는 것입니다. 그것은 답을 Unity Catalog 볼륨의 특정 소스 파일에 매핑하여 사용자가 정확성을 검증할 수 있게 합니다.

### D4. 품질 루프 (리뷰 앱 및 평가)

이것이 Agent Bricks의 차별점이다.

* **리뷰 앱:** 이해관계자(SME)가 상담원과 채팅하며 피드백(좋아요/싫어요/편집)을 제공할 수 있는 기본 내장 UI입니다.  
* **LLM 판사:** 시스템은 상호작용 트레이스에 대해 "LLM 판사"를 사용하기 위해 **Mosaic AI Agent Evaluation**을 사용합니다. 이 심사위원들은 '충실함'(모델이 환각을 본 것인가?)과 '정확성' 같은 지표를 평가합니다.  
* **최적화:** 에이전트 브릭스는 수집된 피드백을 바탕으로 시스템 지침이나 구성 업데이트를 제안하여 성능 지표를 개선합니다.




## E. 요약

**Agent Bricks를 활용한 지식 어시스턴트**는 AI 구성 요소를 수동으로 설계하는 방식에서 AI 결과물을 관리하는 방식으로의 전환을 의미합니다. 선언적 접근 방식을 활용함으로써 팀은 기업 데이터에 기반한 RAG(검색 강화 생성) 시스템을 단 몇 분 만에 배포할 수 있습니다.

**핵심 내용:**

1. **구성보다 최적화:** Agent Bricks는 모델과 검색 매개변수 선택을 자동화하여 비용과 품질 간의 균형을 맞춥니다.  
2. **통합 아키텍처:** Unity catalog 볼륨, ai\_parse\_document, 및 AI Search를 자동으로 조정합니다.
3. **피드백 기반:** 시스템은 Review App과 Agent Learning from Human Feedback(ALHF)을 통해 시간이 지남에 따라 지속적으로 개선되며, 전문가의 피드백을 시스템 개선으로 전환합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>